# APFL Client

> The core abstraction for `APFL` client.

In [ ]:
#| default_exp clients.apfl

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| export
from fastcore.utils import *
from fastcore.all import *
import copy

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

from fedai.clients.base_client import BaseClient
from fedai.utils.registery import AlgorithmRegistry

<torch._C.Generator>

In [ ]:
#| export
@AlgorithmRegistry.register_client("apfl")
class ClientAPFL(BaseClient):
    def __init__(self,
                 id, # Unique identifier for the client
                 cfg, # A configuration object containing all the necessary parameters for training and evaluation.
                 train_loader, 
                 test_loader,
                 state, # A dictionary containing the model, optimizer and any changing component needed.
                 criterion,
                 device= None,
                 t= 0,
                 **kwargs # extra client-specific parameters (that cannot be passed in state and cfg), can be passed as here.
                 ):  
                 
        super().__init__(id, cfg, train_loader, test_loader, state, criterion, device, t, **kwargs)
        

### Training

In [ ]:
#| export
def alpha_update(self: ClientAPFL):
    grad_alpha = 0
    for l_params, p_params in zip(self.model.parameters(), self.model_per.parameters()):
        dif = p_params.data - l_params.data
        grad = self.alpha * p_params.grad.data + (1-self.alpha) * l_params.grad.data
        grad_alpha += dif.view(-1).T.dot(grad.view(-1))
    
    grad_alpha += 0.02 * self.alpha
    self.alpha = self.alpha - self.cfg.optimizer.lr * grad_alpha
    self.alpha = np.clip(self.alpha.item(), 0.0, 1.0)

In [ ]:
#| export
def fit(self: ClientAPFL):
    self.model = self.model.to(self.device)
    self.model.train()
    for _ in range(self.cfg.local_epochs):
        for i, batch in enumerate(self.train_loader):
            batch = self.send_to_device(batch)
            X, y = batch[self.data_key], batch[self.label_key]
            output = self.model(X)
            loss = self.criterion(output, y)
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()

            output_per = self.model_per(X)
            loss_per = self.criterion(output_per, y)
            self.optimizer_per.zero_grad()
            loss_per.backward()
            self.optimizer_per.step()

            self.alpha_update()
            
    for lp, p in zip(self.model_per.parameters(), self.model.parameters()):
        lp.data = (1 - self.alpha) * p + self.alpha * lp

### Saving

In [ ]:
#| export
@patch
def save_state(self: ClientAPFL, save_to_disk= False):

    client_state = {
        "model": self.model.state_dict(),
        "model_per": self.model_per.state_dict(),
        'optimizer': self.optimizer.state_dict(),
        "optimizer_per": self.optimizer_per.state_dict(),
        "alpha": self.alpha
    }

    if save_to_disk:
        state_path = os.path.join(self.cfg.server.save_dir, str(self.t), f"local_output_{self.id}", "state.pth")
        if not os.path.exists(os.path.dirname(state_path)):
            os.makedirs(os.path.dirname(state_path))
        torch.save(client_state, state_path)

    return client_state
        

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()